In [ ]:
from __future__ import annotations

%matplotlib inline
import io
import base64
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from scipy.optimize import curve_fit
from scipy import stats

from cycler import cycler

plt.ioff()  # figures are shown explicitly through Output widgets

PlotStyle = {
    'axes.ymargin': 0.1,
    'legend.frameon': False,
    'xaxis.labellocation': 'right',
    'yaxis.labellocation': 'top',

    'axes.formatter.limits': (-2, 3),

    # 2. ax.ticklabel_format(useMathText=True)
    'axes.formatter.use_mathtext': True,

    # 3. ax.minorticks_on()
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,

    'xtick.major.size': 6,
    'ytick.major.size': 6,

    'xtick.labelsize': 14,
    'ytick.labelsize': 14,

    'xtick.direction': 'in',
    'ytick.direction': 'in',

    'xtick.top': True,
    'ytick.right': True,

    # 2. ax.tick_params(which='minor', length=3, direction='in', right=True, top=True)
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,

    # Note: Direction, top, and right settings automatically apply to minor ticks
    # when set globally, but you can explicitly ensure they mirror major ticks.
    'xtick.minor.top': True,
    'ytick.minor.right': True,

    'axes.xmargin': 0.0,

    'legend.title_fontsize': 16,
    'legend.fontsize': 14,
    'axes.labelsize': 17,
    'axes.titlesize': 16,

    'legend.handleheight': 1,
    'legend.handlelength': 1.2,

    'savefig.dpi': 300,

    'axes.prop_cycle': (
        cycler('color', ["#e6091c", "#e6091c","#e6091c","#e6091c","#e6091c","#e6091c"]) +
        cycler('ls', ['-', '--', '-.', ':', '-', '--'])
    ),
    'savefig.transparent': True,
    'savefig.bbox': 'tight',
}

plt.style.use(PlotStyle)


In [ ]:
# ---------------------------------------------------------------------
# Deployment switch.
#
# While True, both tabs show a "Fake data" button that fills the table
# with simulated data (Poisson-distributed counts, built from the
# formula currently selected in the Function drop-down) - handy while
# designing/testing the activity, or for a live demo.
#
# Set this to False before deploying the app for the real event (e.g. on
# Binder) to remove that button entirely.
# ---------------------------------------------------------------------
ENABLE_FAKE_DATA = False


def _panel(html):
    """Wrap explanatory HTML in a light panel (no widget border needed)."""
    return f"<div style='background:#f7f8fa; border-radius:6px; padding:10px 12px'>{html}</div>"


# ---------------------------------------------------------------------
# All user-facing text lives here, so it can be edited/translated in one
# place without touching the widget-building code below.
# ---------------------------------------------------------------------
TEXTS = {
    "app_title": "International Cosmic Day",
    "app_subtitle": "Data analysis application",

    "tab_titles": {"count": "Counting analysis", "angular": "Angular analysis"},

    "column_labels": {
        "count": ("value", "count"),
        "angular": ("theta / cos(theta)", "count"),
    },

    "panels": {
        "count": _panel("""
<h4>Counting analysis</h4>
<p>Enter, in the table on the left, a series of <b>(value, count)</b> pairs
(for example, the reading of a measured quantity and how many times that
reading occurred). Press <b>Plot</b> to turn the table into a histogram
with unit-width bins spanning the full range of values you entered - bins
you did not fill in are shown with zero count.</p>
<p>Once a histogram is shown, pick a model from the <b>Function</b>
drop-down (or write your own using <code>x</code> and parameters
<code>p0, p1, ...</code>), give starting values for the parameters, and
press <b>Fit</b>. The best-fit curve is overlaid on the plot, together
with a shaded 1&sigma; uncertainty band (untick the checkbox to hide it),
and the fitted parameters (with uncertainties) are printed at the top of
the log below.</p>
"""),
        "angular": _panel("""
<h4>Angular analysis</h4>
<p>Enter, in the table on the left, the angular position (either
<code>theta</code> or <code>cos(theta)</code> - be consistent) and the raw
number of counts observed at that position. Use the two fields below the
table to give the number of individual measurements taken at each angle,
and the estimated uncertainty on the angle itself.</p>
<p>Pressing <b>Plot</b> shows the counts vs. angle, with error bars.</p>
<p>Pick a model, then press <b>Fit</b> to overlay the best fit and its
1&sigma; band (untick the checkbox to hide the band).</p>
"""),
    },

    "labels": {
        "n_meas": "N measurements:",
        "angle_err": "Angle error:",
        "function": "Function:",
        "expr": "f(x,p) =",
        "init_params": "Init. params:",
        "show_band": "Show 1\u03c3 band",
        "log_title": "Log",
    },

    "buttons": {
        "plot": "Plot",
        "fit": "Fit",
        "fake_data": "\U0001F3B2 Fake data",
        "add_row": "+ row",
        "remove_row": "x",
        "download_pdf": "\u2b07 PDF",
        "download_jpg": "\u2b07 JPG",
    },

    "tooltips": {
        "fake_data": "Fill the table with simulated (Poisson) data for the "
                      "selected function - demo/testing only.",
        "download_pdf": "Download the current plot as a high-resolution PDF.",
        "download_jpg": "Download the current plot as a high-resolution JPG.",
        "show_band": "Show/hide the \u00b11\u03c3 uncertainty band around the fit, "
                      "computed from the fit's covariance matrix.",
    },

    "log": {
        "plot_ok": "Plot generated.",
        "plot_need_rows": "Need at least 2 rows of data.",
        "plot_error": "Error while plotting: {err}",
        "fit_no_plot": "No plot available &mdash; press Plot first.",
        "fit_error": "Error while fitting: {err}",
        "fit_result": "Fit result: {params}",
        "fake_ok": "Injected simulated (fake) data.",
        "fake_no_profile": "No fake-data profile defined for this function.",
        "fake_error": "Error while generating fake data: {err}",
        "download_no_plot": "No plot available to download &mdash; press Plot first.",
        "download_error": "Error while preparing download: {err}",
    },
}


In [ ]:
class EditableTable:
    """A small spreadsheet-like widget: fixed number of columns,
    variable number of rows, rows can be added / removed interactively."""

    def __init__(self, col1_label, col2_label, col2_type=float, n_init=6):
        self.col2_type = col2_type
        self.rows = []  # list of dicts: {'box', 'w1', 'w2'}

        header = widgets.HBox([
            widgets.HTML(f"<b>{col1_label}</b>", layout=widgets.Layout(width='120px')),
            widgets.HTML(f"<b>{col2_label}</b>", layout=widgets.Layout(width='120px')),
            widgets.HTML("", layout=widgets.Layout(width='40px')),
        ], layout=widgets.Layout(flex='0 0 auto'))

        self.rows_box = widgets.VBox([], layout=widgets.Layout(
            flex='1 1 auto', overflow_y='auto', min_height='0px'))

        add_btn = widgets.Button(description=TEXTS["buttons"]["add_row"], button_style='info',
                                  layout=widgets.Layout(width='70px', flex='0 0 auto'))
        add_btn.on_click(lambda b: self.add_row())

        for _ in range(n_init):
            self.add_row()

        self.widget = widgets.VBox(
            [header, self.rows_box, add_btn],
            layout=widgets.Layout(flex='1 1 auto', display='flex',
                                   flex_flow='column', min_height='0px'))

    def add_row(self, v1=0.0, v2=0):
        Widget2 = widgets.FloatText if self.col2_type == float else widgets.IntText
        w1 = widgets.FloatText(value=v1, layout=widgets.Layout(width='120px'))
        w2 = Widget2(value=v2, layout=widgets.Layout(width='120px'))
        rm_btn = widgets.Button(description=TEXTS["buttons"]["remove_row"], button_style='danger',
                                 layout=widgets.Layout(width='40px'))
        row_box = widgets.HBox([w1, w2, rm_btn])
        entry = {'box': row_box, 'w1': w1, 'w2': w2}

        def remove(b, entry=entry):
            if entry in self.rows:
                self.rows.remove(entry)
                self.rows_box.children = tuple(e['box'] for e in self.rows)

        rm_btn.on_click(remove)
        self.rows.append(entry)
        self.rows_box.children = tuple(e['box'] for e in self.rows)

    def get_data(self):
        v1 = np.array([e['w1'].value for e in self.rows], dtype=float)
        v2 = np.array([e['w2'].value for e in self.rows], dtype=float)
        return v1, v2

    def set_data(self, pairs):
        """Replace all rows with the given (v1, v2) pairs - used by the
        fake-data generator."""
        self.rows = []
        self.rows_box.children = ()
        for v1, v2 in pairs:
            self.add_row(v1, v2)


def make_logger(log_out):
    """Returns a log(msg, level) function for one panel. Newest message is
    always shown first (inserted at the top), so older ones scroll out of
    view rather than pushing new ones down."""
    messages = []

    def log(msg, level="info"):
        color = {"info": "#333333", "error": "#b00020", "result": "#0a7a2f"}.get(level, "#333333")
        messages.insert(0, (
            f"<div style='color:{color}; font-family:monospace; font-size:0.85em; "
            f"padding:2px 0; border-bottom:1px solid #eee'>{msg}</div>"
        ))
        with log_out:
            clear_output(wait=True)
            display(HTML("".join(messages)))

    return log


def make_fit_func(expr):
    """Build f(x, *p) from a text expression using x and p0, p1, ... as the
    free parameters. numpy (as np, plus a few bare names) and a few
    scipy.stats distributions (norm, poisson, moyal) are exposed."""
    allowed = {k: getattr(np, k) for k in dir(np) if not k.startswith("_")}
    allowed.update({"norm": stats.norm, "poisson": stats.poisson, "moyal": stats.moyal,
                    "exp": np.exp, "cos": np.cos, "sin": np.sin, "round": np.round, "np": np})

    def f(x, *p):
        ns = dict(allowed)
        ns['x'] = x
        for i, val in enumerate(p):
            ns[f'p{i}'] = val
        return eval(expr, {"__builtins__": {}}, ns)

    return f


def fit_uncertainty_band(f, xx, popt, pcov):
    """1-sigma uncertainty band of f(xx, *popt), propagated from the fit's
    covariance matrix via a numerical (finite-difference) Jacobian:
    Var[f(x)] = J(x) . pcov . J(x)^T."""
    n = len(popt)
    f0 = f(xx, *popt)
    J = np.zeros((len(xx), n))
    for i in range(n):
        step = 1e-6 * (abs(popt[i]) if popt[i] != 0 else 1.0)
        p_shift = list(popt)
        p_shift[i] += step
        J[:, i] = (f(xx, *p_shift) - f0) / step
    var = np.einsum('ij,jk,ik->i', J, pcov, J)
    return f0, np.sqrt(np.clip(var, 0, None))


def make_download_html(data_bytes, filename, mime):
    """A hidden auto-clicking download link (the standard Voila/Jupyter
    trick for triggering a browser download from a button click), plus a
    visible fallback link in case the browser blocks the auto-click."""
    b64 = base64.b64encode(data_bytes).decode()
    uid = "dl_" + "".join(ch if ch.isalnum() else "_" for ch in filename) + f"_{np.random.randint(1e9)}"
    return (
        f'<a id="{uid}" download="{filename}" href="data:{mime};base64,{b64}" '
        f'style="font-size:0.85em">\u2b07 if the download did not start, click here for {filename}</a>'
        f'<script>(function(){{var l=document.getElementById("{uid}"); if(l){{l.click();}}}})();</script>'
    )


# ---------------------------------------------------------------------
# Function library shown in the "Function" drop-down of each tab.
#
# To add a new function: add one more entry to the relevant dict below.
#   key          -> label shown in the drop-down
#   expr         -> text evaluated by make_fit_func: use x, p0, p1, ...
#   init         -> comma-separated default initial parameter values (fit)
#   latex        -> LaTeX shown next to the drop-down (rendered with MathJax)
#   fake_x       -> (optional) x-values used to generate fake/demo data
#   fake_params  -> (optional) comma-separated "true" parameters used to
#                   generate fake/demo data (only used if ENABLE_FAKE_DATA)
# ---------------------------------------------------------------------
FUNCTION_LIBRARY = {
    "count": {
        "Gaussian": {
            "expr": "p0*exp(-(x-p1)**2/(2*p2**2))",
            "init": "1, 0, 1",
            "latex": r"$p_0\, e^{-(x-p_1)^2/(2 p_2^2)}$",
            "fake_x": list(range(-6, 7)),
            "fake_params": "60, 0, 2",
        },
        "Moyal (approximates Landau distribution)": {
            "expr": "p0*moyal.pdf(x, p1, p2)",
            "init": "1, 0, 1",
            "latex": r"$p_0 \cdot \mathrm{Moyal}(x;\,p_1,p_2)$",
            "fake_x": list(range(-3, 13)),
            "fake_params": "400, 0, 1",
        },
        "Poisson": {
            "expr": "p0*poisson.pmf(round(x), p1)",
            "init": "1, 1",
            "latex": r"$p_0 \cdot \mathrm{Pois}(x;\,p_1)$",
            "fake_x": list(range(0, 13)),
            "fake_params": "300, 4",
        },
    },
    "angular": {
        "x = \u03b8 angle (\u00b0)": {
            "expr": "p0*cos(np.deg2rad(x))**p1",
            "init": "1, 2",
            "latex": r"$I_0\times\cos^n\theta \quad (x=\theta)$",
            "fake_x": list(np.linspace(0, 80, 9)),
            "fake_params": "80, 2",
        },
        # "I_0 * x**n [x = cos(theta)]": {
        #     "expr": "p0*x**p1",
        #     "init": "1, 2",
        #     "latex": r"\(I_0\times\cos^n\theta \quad (x=\cos\theta)\)",
        # },
    },
}


In [ ]:
def plot_counts(value, count):
    """Histogram with unit-width bins spanning [min, max] of the entered
    values; any integer value not present in the table gets count = 0."""
    order = np.argsort(value)
    v_raw, c_raw = value[order], count[order]

    vmin, vmax = int(np.floor(v_raw.min())), int(np.ceil(v_raw.max()))
    xs = np.arange(vmin, vmax + 1)

    counts_map = {}
    for vv, cc in zip(v_raw, c_raw):
        key = int(round(vv))
        counts_map[key] = counts_map.get(key, 0) + cc
    ys = np.array([counts_map.get(int(xv), 0) for xv in xs], dtype=float)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    # Calculate bin width assuming uniform spacing
    dx = xs[1] - xs[0]

    # Extend xs by half a bin width on each end to preserve full bin coverage with where='mid'
    xs_padded = np.concatenate([[xs[0] - dx], xs, [xs[-1] + dx]])
    ys_padded = np.concatenate([[0], ys, [0]])

    # Plot using padded arrays
    ax.step(xs_padded, ys_padded, where="mid", alpha=0.85, color='k', label="Data")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    ax.set_title("Cosmic Ray Flux", loc='left', color='gray')
    fig.tight_layout()
    return fig, ax, xs.astype(float), ys


def plot_angular(x, count, n_meas, angle_err):
    """Rate vs angle, with Poisson y-errors and constant x-errors."""
    order = np.argsort(x)
    xs, c = x[order], count[order]
    n_meas = n_meas if n_meas and n_meas > 0 else 1

    rate = c
    # Poisson error; floor at 1 count so zero-count bins don't get a
    # zero (i.e. infinite-weight) uncertainty in the fit.
    rate_err = np.sqrt(c/n_meas)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.errorbar(xs, rate, yerr=rate_err,
                xerr=angle_err if angle_err and angle_err > 0 else None,
                fmt='o', capsize=0, color='k', label='Data')
    ax.set_xlabel("Angle $\\theta$ [${}^\\circ$]")
    ax.set_ylabel("Rate [counts / fixed time]")
    ax.set_title("Cosmic Ray Flux", loc='left', color='gray')
    fig.tight_layout()
    return fig, ax, xs, rate, rate_err


In [ ]:
def build_analysis_tab(mode):
    """mode: 'count' or 'angular'. Returns the 3-column widgets.HBox for one tab.

    Layout fills all the height it is given: the outer HBox and the three
    VBox columns stretch to 100%, with the table rows / image / log panels
    as the flexible (scrolling) parts and the controls as fixed-size."""

    col1_label, col2_label = TEXTS["column_labels"][mode]
    std_text = TEXTS["panels"][mode]
    L = TEXTS["labels"]
    B = TEXTS["buttons"]
    TT = TEXTS["tooltips"]
    LOG = TEXTS["log"]

    func_options = FUNCTION_LIBRARY[mode]

    table = EditableTable(col1_label, col2_label, col2_type=int, n_init=6)

    if mode == 'angular':
        n_meas_w = widgets.IntText(value=1, description=L["n_meas"],
                                    style={'description_width': 'initial'},
                                    layout=widgets.Layout(width='230px'))
        angle_err_w = widgets.FloatText(value=0.0, description=L["angle_err"],
                                         style={'description_width': 'initial'},
                                         layout=widgets.Layout(width='230px'))
        extra_inputs = widgets.VBox([n_meas_w, angle_err_w],
                                     layout=widgets.Layout(flex='0 0 auto'))
    else:
        n_meas_w = angle_err_w = None
        extra_inputs = widgets.VBox([], layout=widgets.Layout(flex='0 0 auto'))

    func_dropdown = widgets.Dropdown(options=list(func_options.keys()),
                                      description=L["function"],
                                      style={'description_width': 'initial'},
                                      layout=widgets.Layout(width='55%'))
    latex_display = widgets.HTMLMath(value="", layout=widgets.Layout(width='40%', padding='5px 0 5px 10px'))
    func_expr_w = widgets.Text(description=L["expr"],
                                style={'description_width': 'initial'},
                                layout=widgets.Layout(width='95%'))
    p0_w = widgets.Text(description=L["init_params"],
                         style={'description_width': 'initial'},
                         layout=widgets.Layout(width='95%'))
    band_checkbox = widgets.Checkbox(value=True, description=L["show_band"],
                                      tooltip=TT["show_band"],
                                      layout=widgets.Layout(width='95%'))

    def on_func_change(change):
        spec = func_options[change['new']]
        func_expr_w.value = spec['expr']
        p0_w.value = spec['init']
        latex_display.value = spec['latex']

    func_dropdown.observe(on_func_change, names='value')
    on_func_change({'new': func_dropdown.value})

    plot_btn = widgets.Button(description=B["plot"], button_style='primary',
                               layout=widgets.Layout(width='90px'))
    fit_btn = widgets.Button(description=B["fit"], button_style='success',
                              disabled=True, layout=widgets.Layout(width='90px'))
    pdf_btn = widgets.Button(description=B["download_pdf"], tooltip=TT["download_pdf"],
                              disabled=True, layout=widgets.Layout(width='90px'))
    jpg_btn = widgets.Button(description=B["download_jpg"], tooltip=TT["download_jpg"],
                              disabled=True, layout=widgets.Layout(width='90px'))

    action_buttons = [plot_btn, fit_btn]

    if ENABLE_FAKE_DATA:
        fake_btn = widgets.Button(description=B["fake_data"], button_style='warning',
                                   tooltip=TT["fake_data"],
                                   layout=widgets.Layout(width='120px'))
        action_buttons.append(fake_btn)
    else:
        fake_btn = None

    func_selector = widgets.HBox(
        [func_dropdown, latex_display],
        layout=widgets.Layout(width='100%', align_items='center')
    )

    controls = widgets.VBox(
        [extra_inputs, widgets.HTML("<hr>"), func_selector,
         func_expr_w, p0_w, band_checkbox, widgets.HBox(action_buttons),
         widgets.HBox([pdf_btn, jpg_btn])],
        layout=widgets.Layout(flex='0 0 auto')
    )

    col1 = widgets.VBox(
        [table.widget, controls],
        layout=widgets.Layout(width='33%', height='100%', display='flex',
                               flex_flow='column', min_height='0px',
                               border_right='1px solid #e2e2e2', padding='0 10px')
    )

    image_out = widgets.Output(layout=widgets.Layout(
        width='100%', flex='1 1 auto', overflow='auto'))
    col2 = widgets.VBox(
        [image_out],
        layout=widgets.Layout(width='33%', height='100%', display='flex',
                               flex_flow='column', min_height='0px',
                               border_right='1px solid #e2e2e2', padding='0 10px')
    )

    text_out = widgets.HTML(std_text, layout=widgets.Layout(
        width='100%', flex='0 1 auto', overflow='auto'))
    log_out = widgets.Output(layout=widgets.Layout(
        width='100%', flex='1 1 auto', overflow='auto', min_height='0px'))
    col3 = widgets.VBox(
        [text_out, widgets.HTML(f"<b>{L['log_title']}</b>"), log_out],
        layout=widgets.Layout(width='33%', height='100%', display='flex',
                               flex_flow='column', min_height='0px', padding='0 10px')
    )

    log = make_logger(log_out)
    state = {'has_plot': False, 'fig': None, 'ax': None, 'x': None, 'y': None, 'yerr': None,
             'fit_xx': None, 'fit_yy': None, 'fit_band': None}

    def _redraw():
        with image_out:
            clear_output(wait=True)
            display(state['fig'])

    def _clear_fit_artists(ax):
        for line in list(ax.lines):
            if getattr(line, '_is_fit_line', False):
                line.remove()
        for coll in list(ax.collections):
            if getattr(coll, '_is_fit_band', False):
                coll.remove()

    def do_plot(b):
        fit_btn.disabled = True
        pdf_btn.disabled = True
        jpg_btn.disabled = True
        state['has_plot'] = False
        state['fit_xx'] = state['fit_yy'] = state['fit_band'] = None
        with image_out:
            clear_output(wait=True)
        try:
            v1, v2 = table.get_data()
            if len(v1) < 2:
                raise ValueError(LOG["plot_need_rows"])
            if mode == 'count':
                fig, ax, x, y = plot_counts(v1, v2)
                yerr = None
            else:
                fig, ax, x, y, yerr = plot_angular(
                    v1, v2, n_meas_w.value, angle_err_w.value)
            with image_out:
                display(fig)
            state.update(has_plot=True, fig=fig, ax=ax, x=x, y=y, yerr=yerr)
            fit_btn.disabled = False
            pdf_btn.disabled = False
            jpg_btn.disabled = False
            log(LOG["plot_ok"], "info")
        except Exception as e:
            log(LOG["plot_error"].format(err=e), "error")

    def do_fit(b):
        if not state['has_plot']:
            log(LOG["fit_no_plot"], "error")
            return
        try:
            f = make_fit_func(func_expr_w.value)
            p0 = [float(s) for s in p0_w.value.split(',') if s.strip() != ""]
            x, y, sigma = state['x'], state['y'], state['yerr']
            popt, pcov = curve_fit(f, x, y, p0=p0, sigma=sigma,
                                    absolute_sigma=sigma is not None)
            perr = np.sqrt(np.diag(pcov))

            ax = state['ax']
            _clear_fit_artists(ax)

            dxx = x[1] - x[0]
            xx = np.linspace(x.min() - dxx, x.max() + dxx, 300)
            yy, band = fit_uncertainty_band(f, xx, popt, pcov)
            state['fit_xx'], state['fit_yy'], state['fit_band'] = xx, yy, band

            if band_checkbox.value:
                band_artist = ax.fill_between(xx, yy - band, yy + band, color='red',
                                               alpha=0.2, linewidth=0, label='1$\\sigma$ band')
                band_artist._is_fit_band = True
            line, = ax.plot(xx, yy, lw=2, c='r', label='Fit')
            line._is_fit_line = True
            ax.legend()

            _redraw()

            msg = LOG["fit_result"].format(params=", ".join(
                f"p{i} = {v:.4g} &plusmn; {e:.2g}" for i, (v, e) in enumerate(zip(popt, perr))))
            log(msg, "result")
        except Exception as e:
            log(LOG["fit_error"].format(err=e), "error")

    def on_band_toggle(change):
        if state['fit_yy'] is None:
            return
        ax = state['ax']
        _clear_fit_artists(ax)
        if change['new']:
            band_artist = ax.fill_between(state['fit_xx'], state['fit_yy'] - state['fit_band'],
                                           state['fit_yy'] + state['fit_band'], color='red',
                                           alpha=0.2, linewidth=0, label='1$\\sigma$ band')
            band_artist._is_fit_band = True
        line, = ax.plot(state['fit_xx'], state['fit_yy'], lw=2, label='Fit')
        line._is_fit_line = True
        ax.legend()
        _redraw()

    band_checkbox.observe(on_band_toggle, names='value')

    def do_fake_data(b):
        spec = func_options[func_dropdown.value]
        fx, fp = spec.get('fake_x'), spec.get('fake_params')
        if fx is None or fp is None:
            log(LOG["fake_no_profile"], "error")
            return
        try:
            x_arr = np.array(fx, dtype=float)
            f_true = make_fit_func(spec['expr'])
            params = [float(s) for s in fp.split(',') if s.strip() != ""]
            expected = np.clip(f_true(x_arr, *params), 0, None)
            if mode == 'angular':
                n = n_meas_w.value if n_meas_w.value else 100
                n_meas_w.value = n
                counts = np.random.poisson(expected)
            else:
                counts = np.random.poisson(expected)
            table.set_data(list(zip(x_arr, counts)))
            log(LOG["fake_ok"], "info")
            do_plot(None)
        except Exception as e:
            log(LOG["fake_error"].format(err=e), "error")

    def _download(fmt, mime):
        if not state['has_plot']:
            log(LOG["download_no_plot"], "error")
            return
        try:
            buf = io.BytesIO()
            savefig_kwargs = dict(dpi=600, bbox_inches='tight')
            if fmt == 'jpg':
                savefig_kwargs.update(facecolor='white', transparent=False)
            state['fig'].savefig(buf, format=fmt, **savefig_kwargs)
            filename = f"{mode}_analysis.{fmt}"
            log(make_download_html(buf.getvalue(), filename, mime), "info")
        except Exception as e:
            log(LOG["download_error"].format(err=e), "error")

    plot_btn.on_click(do_plot)
    fit_btn.on_click(do_fit)
    pdf_btn.on_click(lambda b: _download('pdf', 'application/pdf'))
    jpg_btn.on_click(lambda b: _download('jpg', 'image/jpeg'))
    if fake_btn is not None:
        fake_btn.on_click(do_fake_data)

    return widgets.HBox(
        [col1, col2, col3],
        layout=widgets.Layout(width='100%', height='100%', display='flex', flex_flow='row')
    )


In [ ]:
# Best-effort CSS so the app uses the full height of the browser window
# rather than a fixed pixel size (exact behaviour depends on the Voila
# template/version).
_ = display(HTML("""
<style>
  html, body { height: 100%; }
  .jp-Notebook, #voila-container, .jp-OutputArea, .jp-OutputArea-output,
  .jp-Cell, .jp-Cell-outputWrapper { overflow: visible; }
  .widget-tab, .p-TabBar-content { flex: 0 0 auto; }
</style>
"""))

tab = widgets.Tab(
    children=[build_analysis_tab('count'), build_analysis_tab('angular')],
    layout=widgets.Layout(flex='1 1 auto', height='100%', min_height='0px')
)
tab.set_title(0, TEXTS["tab_titles"]["count"])
tab.set_title(1, TEXTS["tab_titles"]["angular"])


def _load_logo(path):
    """Falls back to a tiny transparent placeholder if the logo file is
    missing, so the app still runs (e.g. when testing without /bin)."""
    try:
        with open(path, 'rb') as fh:
            return fh.read()
    except OSError:
        return base64.b64decode(
            'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+A8AAQUB'
            'AScY42YAAAAASUVORK5CYII='
        )


logoOcra = widgets.Image(value=_load_logo('bin/infn-ocra.png'), format='png', width=150)
logoUnige = widgets.Image(value=_load_logo('bin/unige.jpg'), format='jpg', width=150)

pageTitle = widgets.HTML(
    f"<h1>{TEXTS['app_title']}</h1><h2>{TEXTS['app_subtitle']}</h2>"
)

space = widgets.Box(layout=widgets.Layout(width='20px'))

header = widgets.HBox(
    [
        widgets.Box([pageTitle], layout=widgets.Layout(width='100%', display='flex')),
        logoUnige,
        space,
        logoOcra
    ],
    layout=widgets.Layout(align_items='center', margin='0px 0px')
)

app = widgets.VBox(
    [header, tab],
    layout=widgets.Layout(height='92vh', display='flex', flex_flow='column')
)

_ = display(app)
